In [2]:
!pip install pulp
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import pulp
import os

## 1. 전처리 및 제약 조건

In [4]:
elder_df = pd.read_csv("center_elder.csv", encoding="cp949")
vacant_df = pd.read_csv("/content/부산_빈집실태_전처리.csv")
r_count_df = pd.read_csv("R_count.csv", encoding="cp949")
slope_df = pd.read_csv("slope.csv", encoding="cp949")
price_df = pd.read_csv("/content/부산_법정동별_공시지가_전처리.csv")

In [5]:
#빈집 1,2등급이 300개 이상인 군구만 추출
filtered_vacant_df = vacant_df[vacant_df['빈집_1,2등급'] >= 300]

In [6]:
#인프라 컬럼
facilities_cols = [
    "EMD_CD",
    "NUMPOINTS_경로당", "NUMPOINTS_종합병원", "NUMPOINTS_BUS",
    "NUMPOINTS_노인복지관", "NUMPOINTS_노인교실", "NUMPOINTS_대형점포",
    "NUMPOINTS_시장", "NUMPOINTS_SUBWAY"
]
r_count_filtered = r_count_df[facilities_cols]

In [7]:
#고령 + 인프라 + 경사도
merged = pd.merge(elder_df, r_count_filtered, on='EMD_CD', how='inner')
merged = pd.merge(merged, slope_df[['EMD_CD', 'slope_mean']], on='EMD_CD', how='inner')

In [8]:
#고령 + 인프라 + 경사도 + 필지수
merged = pd.merge(
    merged,
    price_df[['구군', '법정동', '필지수']],
    left_on=['고령인구비율_읍면동_구', 'EMD_NM'],
    right_on=['구군', '법정동'],
    how='left'
).drop(columns=['구군', '법정동'], errors='ignore')

In [9]:
# 300개 이상 구군의 빈집 데이터만 법정동에 그대로 복사 할당 (나머지 구 소속 동은 자동 탈락)
final_merged_df = pd.merge(
    merged,
    filtered_vacant_df[['구군', '빈집_1,2등급']],
    left_on='고령인구비율_읍면동_구',
    right_on='구군',
    how='inner' # inner 매핑으로 조건 미달 구 일괄 제거
).drop(columns=['구군'], errors='ignore')

In [10]:
final_merged_df['elder_cnt'] = final_merged_df['고령인구비율_읍면동_노령전체']

In [11]:
final_merged_df.to_csv("ahp_mclp_input_data.csv", index=False, encoding="utf-8-sig")

## 2. min-max 정규화 + AHP 적합도 계산

In [12]:
df = pd.read_csv("ahp_mclp_input_data.csv")

In [13]:
FACILITIES = [
    "NUMPOINTS_경로당", "NUMPOINTS_종합병원", "NUMPOINTS_BUS",
    "NUMPOINTS_노인복지관", "NUMPOINTS_노인교실", "NUMPOINTS_대형점포",
    "NUMPOINTS_시장", "NUMPOINTS_SUBWAY"
]

In [14]:
#AHP 가중치 설정
_H = 0.46   #병원
_T = 0.28   #교통
_C = 0.17   #생활편의
_W = 0.09   #복지

In [15]:
AHP_WEIGHTS = {
    "NUMPOINTS_경로당":   _W/3,    # 0.0300
    "NUMPOINTS_노인복지관": _W/3,    # 0.0300
    "NUMPOINTS_노인교실": _W/3,    # 0.0300
    "NUMPOINTS_종합병원": _H,      # 0.4600
    "NUMPOINTS_BUS":      _T/2,    # 0.1400
    "NUMPOINTS_SUBWAY":   _T/2,    # 0.1400
    "NUMPOINTS_대형점포": _C/2,    # 0.0850
    "NUMPOINTS_시장":     _C/2,    # 0.0850
}

In [16]:
#빈 배열 생성
ahp_scores = np.zeros(len(df))

In [17]:
#시설별 Min-Max 정규화 연산 수행
for f in FACILITIES:
    v = df[f].values.astype(float)
    rng = v.max() - v.min()

    if rng > 0:
        norm = (v - v.min()) / rng
    else:
        norm = np.zeros_like(v)

    #시설 정규화 값 * 가중치
    ahp_scores += AHP_WEIGHTS[f] * norm

In [18]:
df["ahp_score"] = ahp_scores

In [19]:
df.to_csv("ahp_mclp_ready_data.csv", index=False, encoding="utf-8-sig")

##3. MCLP 기법 적용

In [20]:
df = pd.read_csv("ahp_mclp_ready_data.csv")

In [21]:
#하이퍼파라미터 조건 정의
COVER_DIST = 1000.0            # 커버 반경
VACANT_THRESHOLD = 300         # 빈집 1,2등급 필터 임계값 (이 값 이상이면 후보)
SLOPE_MAX = 15.0               # 후보 경사도 상한 (slope_mean <= 15)
PARCEL_MIN = 500               # 후보 필지수 하한 (필지수 >= 500)
P_LIST = [1, 3, 5]             # 시설 설치 개수 시나리오
LAMBDA_LIST = [0.1, 0.2, 0.3]  # 입지 적합도 반영 비중 시나리오

In [22]:
VACANT_COL = "빈집_1,2등급"
WEIGHT_COL = "elder_cnt"

In [23]:
#최적화 연산에 필요한 기본 배열 추출
coords = df[["coord_x", "coord_y"]].values
weights = df[WEIGHT_COL].values.astype(float)
ahp = df["ahp_score"].values
total_w = weights.sum()
n = len(df)

In [24]:
#후보 선정: (빈집 >= 임계값) AND (경사도 <= SLOPE_MAX) AND (필지수 >= PARCEL_MIN)
cand_mask = (
    (df[VACANT_COL] >= VACANT_THRESHOLD) &
    (df["slope_mean"] <= SLOPE_MAX) &
    (df["필지수"] >= PARCEL_MIN)
)
cand_idx = df[cand_mask].index.values
cand_coords = coords[cand_idx]

In [25]:
print(f"후보 동 수: {len(cand_idx)}개 "
      f"(빈집 >= {VACANT_THRESHOLD}, 경사도 <= {SLOPE_MAX}, 필지수 >= {PARCEL_MIN})")

후보 동 수: 60개 (빈집 >= 300, 경사도 <= 15.0, 필지수 >= 500)


In [26]:
# (후보는 셀 39에서 빈집 임계값으로 확정됨)

In [27]:
# 최적화 연산에 필요한 기본 배열 추출
coords = df[["coord_x", "coord_y"]].values
weights = df[WEIGHT_COL].values.astype(float)
ahp = df["ahp_score"].values
total_w = weights.sum() # 'how='left'' 기반의 정확한 부산 전체 고령인구 수 합산
n = len(df)

# 후보 선정 제약 필터링
cand_mask = (
    (df[VACANT_COL] >= VACANT_THRESHOLD) &
    (df["slope_mean"] <= SLOPE_MAX) &
    (df["필지수"] >= PARCEL_MIN)
)
cand_idx = df[cand_mask].index.values
cand_coords = coords[cand_idx]

print(f"후보 동 수: {len(cand_idx)}개")

# ==============================================================================
# [BUG FIX] 공간 인덱싱(cKDTree)을 루프 외부에서 정석대로 실행
# ==============================================================================
total_tree = cKDTree(coords)
covers_matrix = total_tree.query_ball_point(cand_coords, COVER_DIST)

final_rows = []

for lam in LAMBDA_LIST:
    for p in P_LIST:
        # PuLP 최적화 문제 정의
        prob = pulp.LpProblem("AHP_MCLP", pulp.LpMaximize)

        # 의사결정 이진변수 정의
        x = {j: pulp.LpVariable(f"x_{j}", cat="Binary") for j in range(len(cand_idx))}
        y = {i: pulp.LpVariable(f"y_{i}", cat="Binary") for i in range(n)}

        # 목적함수 식 구성
        coverage_term = pulp.lpSum(weights[i] * y[i] for i in range(n)) / total_w
        fit_term = pulp.lpSum(ahp[cand_idx[j]] * x[j] for j in range(len(cand_idx))) / p
        prob += (1 - lam) * coverage_term + lam * fit_term

        # 제약조건 1: 시설은 정확히 p개만 설치
        prob += pulp.lpSum(x[j] for j in range(len(cand_idx))) == p

        # 제약조건 2: 인덱스 커버리지 매핑 제약
        for i in range(n):
            covering_candidates = [j for j in range(len(cand_idx)) if i in covers_matrix[j]]
            if len(covering_candidates) == 0:
                prob += y[i] == 0
            else:
                prob += y[i] <= pulp.lpSum(x[j] for j in covering_candidates)

        # 솔버 실행
        prob.solve(pulp.PULP_CBC_CMD(msg=0))

        # 최적의 해 추출
        chosen = [cand_idx[j] for j in range(len(cand_idx)) if x[j].value() == 1]
        covered = [i for i in range(n) if y[i].value() == 1]

        # 통계값 계산
        cov_elders = int(sum(weights[i] for i in covered))
        cov_pct = round(100 * cov_elders / total_w, 1)
        fit_avg = round(float(np.mean([ahp[k] for k in chosen])), 3)

        # 상세 데이터 수집 (엑셀 저장용)
        for rank, idx in enumerate(chosen, 1):
            row = df.loc[idx]
            final_rows.append({
                "lambda": lam, "p_scenario": p, "선택순위": rank,
                "EMD_CD": row["EMD_CD"], "동명": row["EMD_NM"],
                "구": row["고령인구비율_읍면동_구"],
                "coord_x": row["coord_x"], "coord_y": row["coord_y"],
                "고령인구": int(row[WEIGHT_COL]),
                "빈집_1,2등급": float(row[VACANT_COL]),
                "필지수": int(row["필지수"]),
                "평균경사도": round(float(row["slope_mean"]), 2),
                "AHP적합도": round(row["ahp_score"], 4),
                "시나리오_커버고령수": cov_elders,
                "시나리오_커버율(%)": cov_pct,
                "시나리오_적합도평균": fit_avg,
            })

In [28]:
# 최종 결과 저장
result_df = pd.DataFrame(final_rows)
output_path = "ahp_mclp_result.csv"
result_df.to_csv(output_path, index=False, encoding="utf-8-sig")

# 결과 일부 확인
result_df.head(10)

,lambda,p_scenario,선택순위,EMD_CD,동명,구,coord_x,coord_y,고령인구,"빈집_1,2등급",필지수,평균경사도,AHP적합도,시나리오_커버고령수,시나리오_커버율(%),시나리오_적합도평균
0,0.1,1,1,26140108,부용동2가,서구,383963.878,280767.438,348,314.0,722,8.52,0.8054,12948,2.9,0.805
1,0.1,3,1,26140108,부용동2가,서구,383963.878,280767.438,348,314.0,722,8.52,0.8054,56736,12.6,0.418
2,0.1,3,2,26230109,당감동,부산진구,384851.746,287541.262,12104,468.0,6971,11.09,0.1447,56736,12.6,0.418
3,0.1,3,3,26290107,용호동,남구,392766.268,281863.008,21505,458.0,5223,12.88,0.3043,56736,12.6,0.418
4,0.1,5,1,26140108,부용동2가,서구,383963.878,280767.438,348,314.0,722,8.52,0.8054,91262,20.2,0.412
5,0.1,5,2,26200110,영선동3가,영도구,386169.877,278524.197,651,305.0,513,3.43,0.6187,91262,20.2,0.412
6,0.1,5,3,26230109,당감동,부산진구,384851.746,287541.262,12104,468.0,6971,11.09,0.1447,91262,20.2,0.412
7,0.1,5,4,26290107,용호동,남구,392766.268,281863.008,21505,458.0,5223,12.88,0.3043,91262,20.2,0.412
8,0.1,5,5,26290106,대연동,남구,390705.346,284716.400,20191,458.0,12681,10.92,0.1888,91262,20.2,0.412
9,0.2,1,1,26140108,부용동2가,서구,383963.878,280767.438,348,314.0,722,8.52,0.8054,12948,2.9,0.805
